# Chapter 12 — Problem Set 2: Solutions

Runnable solutions with explanations. All exercises run **offline** with `MockLLMClient`.

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

from prompt_templates import (
    PromptTemplate, ChainOfThoughtTemplate, ReActTemplate,
    PromptRegistry, default_registry,
)
from llm_clients import MockLLMClient
from evaluation_utils import (
    cosine_match, exact_match,
    PromptEvalHarness, PromptABTester,
    detect_injection,
)
import json
import pandas as pd
from collections import Counter

client = MockLLMClient()

## 1. Self-Consistency — Solution

In [ ]:
import re
cot = ChainOfThoughtTemplate(name='cot', template='')

def self_consistency_answer(question, n_samples=5):
    answers = []
    for seed in range(n_samples):
        text = client.complete(cot.render(input=question), temperature=0.7, seed=seed).text
        m = re.findall(r'-?\d+', text)
        if m:
            answers.append(int(m[-1]))
    if not answers:
        return None
    return Counter(answers).most_common(1)[0][0]

print(self_consistency_answer('Sum of 3 + 4 + 5 = ?', n_samples=7))

## 2. Eval Harness for Summarization — Solution

In [ ]:
DATA_DIR = os.path.join('..', '..', 'datasets')
df = pd.read_csv(os.path.join(DATA_DIR, 'eval_tasks.csv'))
sum_rows = df[df['task_type'] == 'summarization'].to_dict('records')

summ = PromptTemplate(
    name='summ_v1',
    template='Summarize in one sentence.\n\n{{ text }}\nSummary:',
)

def render(inp):
    return summ.render(text=inp)

def grade(pred, ref):
    return {'cosine': cosine_match(pred, ref), 'score': cosine_match(pred, ref)}

harness = PromptEvalHarness(client, render, grade)
report = harness.run(sum_rows)
print('Mean score:', round(report['metrics']['score'], 3))

records = sorted(report['records'], key=lambda r: r.scores['score'])
print('\nWorst:', records[0].input[:60], '->', records[0].prediction[:60], 'score=', round(records[0].scores['score'], 3))
print('Best:', records[-1].input[:60], '->', records[-1].prediction[:60], 'score=', round(records[-1].scores['score'], 3))

## 3. Prompt-Injection Detection — Solution

In [ ]:
path = os.path.join(DATA_DIR, 'injection_examples.txt')
attacks = [l for l in open(path).read().splitlines() if l and not l.startswith('#')]
for a in attacks:
    hits = detect_injection(a)
    print(f'HIT={bool(hits)}: {a[:70]!r}')

# Add a custom pattern (already in defaults but demonstrates the API)
extra = list(detect_injection.__defaults__[0]) if detect_injection.__defaults__ else []
custom = [r'translate.*ignore.*instructions']
print('\nWith custom pattern:')
for a in attacks:
    hits = detect_injection(a, patterns=custom)
    if hits:
        print(f'  Custom HIT: {a!r}')

## 4. A/B Test — Solution

In [ ]:
cls_rows = df[df['task_type'] == 'classification'].to_dict('records')

terse = PromptTemplate(name='terse', template='Sentiment: {{ text }}')
strict = PromptTemplate(name='strict', template=(
    'Classify the sentiment as positive, negative, or neutral. Return only the label.\nText: {{ text }}\nLabel:'
))

def grade_label(pred, ref):
    return {'score': 1.0 if ref.lower() in pred.strip().lower() else 0.0}

def run(prompt):
    h = PromptEvalHarness(client, lambda inp: prompt.render(text=inp), grade_label)
    return [r.scores['score'] for r in h.run(cls_rows)['records']]

a = run(terse)
b = run(strict)
print('A:', a)
print('B:', b)

ab = PromptABTester(n_iterations=2000, seed=0).run(a, b)
print(f'Diff: {ab.diff:+.3f}  CI [{ab.diff_ci_low:+.3f}, {ab.diff_ci_high:+.3f}]  significant={ab.significant}')

## 5. ReAct for Math — Solution

In [ ]:
react = ReActTemplate(
    name='react_math',
    template='',
    tools_description='Calculator[<expr>] -- evaluate a math expression\nFinish[<answer>] -- stop',
)

def calculator(expr):
    try:
        return str(eval(expr, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'ERROR: {e}'

def solve_math(problem, max_steps=4):
    scratchpad = ''
    for step in range(max_steps):
        prompt = react.render(input=problem, scratchpad=scratchpad)
        out = client.complete(prompt).text
        m = re.search(r'(.*?)Action:\s*(\w+)\[(.*?)\]', out, re.DOTALL)
        if not m:
            return None
        thought, tool, arg = m.group(1).strip(), m.group(2), m.group(3)
        if tool == 'Finish':
            return arg
        result = calculator(arg) if tool == 'Calculator' else 'unknown tool'
        scratchpad += f'\nThought:{thought}\nAction: {tool}[{arg}]\nObservation: {result}'
    return None

print(solve_math('Tom has 12 apples. He gives 4 to a friend and buys 7 more. How many does he have now?'))

## 6. Versioned Prompt Registry — Solution

In [ ]:
reg = PromptRegistry()
v1 = PromptTemplate(name='sentiment', version='v1', template='Sentiment: {{ text }}')
v2 = PromptTemplate(
    name='sentiment',
    version='v2',
    template='Classify (positive/negative/neutral). Return only the label.\nText: {{ text }}\nLabel:',
)
reg.register(v1)
reg.register(v2)
print('Listed:', reg.list())

os.makedirs('../../registry', exist_ok=True)
path = '../../registry/ps2_prompts.yaml'
reg.to_yaml(path)
reg2 = PromptRegistry.from_yaml(path)
print('Reloaded:', reg2.list())

for v in ['v1', 'v2']:
    t = reg2.get('sentiment', version=v)
    print(f'  sentiment@{v}: fp={t.fingerprint()}')

---
*Generated by Berta AI*